In [32]:
import pandas as pd

In [33]:
path = "/content/twcs.csv"

In [34]:
def load_dataset(path):
    df = pd.read_csv(path)
    print(f"shape : {df.shape}")
    print(f"columns : {df.columns}")
    print(f"null values : {df.isnull().sum()}")
    print(f"samples : \n {df.head(2)}")

    return df

In [35]:
df = load_dataset(path)

shape : (2811774, 7)
columns : Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')
null values : tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64
samples : 
    tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  


In [36]:
df["inbound"].value_counts()

,count
inbound,
True,1537843
False,1273931


In [37]:
def data_exploration(df):
    print("exploring inbound : ",df["inbound"].value_counts())
    print("exploring authors : ",df["author_id"].value_counts().head(30))

In [38]:
data_exploration(df)

exploring inbound :  inbound
True     1537843
False    1273931
Name: count, dtype: int64
exploring authors :  author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
VerizonSupport      17966
UPSHelp             17817
ATVIAssist          17650
O2                  16212
Safaricom_Care      16077
idea_cares          15724
AskTarget           13218
AirAsiaSupport      12829
BofA_Help           12683
SW_Help             12231
Name: count, dtype: int64


In [47]:
def inspect_amazon_help(df):
    amazon = df[df["author_id"] == "AmazonHelp"]

    print("Amazon tweets:", amazon.shape)

    print("\nInbound/Outbound:")
    print(amazon["inbound"].value_counts())

    return amazon

In [48]:
amazon = inspect_amazon_help(df)

Amazon tweets: (169840, 7)

Inbound/Outbound:
inbound
False    169840
Name: count, dtype: int64


In [50]:
ids = [id for id in amazon['tweet_id']]

In [52]:
amazon_reply_ids = set(ids)

In [64]:
from tqdm.auto import tqdm
def build_conversations(df, amazon_reply_ids):

    # --------------------------------------------------
    # 1. Sort by time
    # --------------------------------------------------
    print("Sorting dataframe...")

    df = df.sort_values("created_at").copy()

    # --------------------------------------------------
    # 2. Create fast lookups
    # --------------------------------------------------
    print("Creating lookup dictionaries...")

    parent_map = dict(
        zip(
            df["tweet_id"],
            df["in_response_to_tweet_id"]
        )
    )

    response_map = dict(
        zip(
            df["tweet_id"],
            df["response_tweet_id"]
        )
    )

    tweet_map = df.set_index("tweet_id").to_dict("index")

    print("Lookup dictionaries created.")

    # --------------------------------------------------
    # 3. Find roots for Amazon tweets
    # --------------------------------------------------
    print("Finding conversation roots...")

    roots = set()

    for amazon_id in tqdm(
        amazon_reply_ids,
        desc="Finding roots"
    ):

        current_id = amazon_id

        while True:

            parent_id = parent_map.get(current_id)

            # No parent -> root found
            if pd.isna(parent_id):
                roots.add(current_id)
                break

            parent_id = int(parent_id)

            # Parent doesn't exist
            if parent_id not in parent_map:
                break

            current_id = parent_id

    print("Unique conversation roots:", len(roots))

    # --------------------------------------------------
    # 4. Traverse conversations forward
    # --------------------------------------------------
    print("Building conversations...")

    conversations = []

    for root_id in tqdm(
        roots,
        desc="Building conversations"
    ):

        conversation_ids = []
        stack = [root_id]
        visited = set()

        while stack:

            current_id = stack.pop()

            if current_id in visited:
                continue

            if current_id not in tweet_map:
                continue

            visited.add(current_id)
            conversation_ids.append(current_id)

            response_ids = response_map.get(current_id)

            if pd.isna(response_ids):
                continue

            # Multiple responses:
            # "722064,722067"
            if isinstance(response_ids, str):
                response_ids = response_ids.split(",")

            else:
                response_ids = [response_ids]

            for response_id in response_ids:

                try:
                    response_id = int(response_id)

                    if response_id not in visited:
                        stack.append(response_id)

                except (ValueError, TypeError):
                    continue

        # --------------------------------------------------
        # 5. Keep only conversations containing Amazon
        # --------------------------------------------------

        if any(
            tweet_id in amazon_reply_ids
            for tweet_id in conversation_ids
        ):

            # Sort chronologically
            conversation_ids.sort(
                key=lambda x: tweet_map[x]["created_at"]
            )

            conversations.append(conversation_ids)

    print(
        "Conversations created:",
        len(conversations)
    )

    return conversations

In [65]:
conversations = build_conversations(df, amazon_reply_ids)

Sorting dataframe...
Creating lookup dictionaries...
Lookup dictionaries created.
Finding conversation roots...


Finding roots:   0%|          | 0/169840 [00:00<?, ?it/s]

Unique conversation roots: 81902
Building conversations...


Building conversations:   0%|          | 0/81902 [00:00<?, ?it/s]

Conversations created: 81902


In [67]:
len(conversations)

81902

In [77]:
from tqdm.auto import tqdm

def save_conversations_txt(
    conversations,
    filename="/content/amazon_conversations.txt"
):

    with open(filename, "w", encoding="utf-8") as f:

        for conversation_id, conversation in enumerate(
            tqdm(
                conversations,
                desc="Saving conversations",
                unit="conversation"
            ),
            start=1
        ):

            f.write("=" * 70 + "\n")
            f.write(f"CONVERSATION {conversation_id}\n")
            f.write("=" * 70 + "\n\n")

            for message in conversation:

                if message["speaker"] == "Customer":
                    f.write("Customer Query:\n")
                else:
                    f.write("Agent Reply:\n")

                f.write(message["text"].strip() + "\n\n")

            f.write("=" * 70 + "\n")
            f.write("END OF CONVERSATION\n")
            f.write("=" * 70 + "\n\n")

    print(f"\nSaved to: {filename}")

In [78]:
formatted_conversations = format_conversations(conversations, df)

Formatting conversations:   0%|          | 0/81902 [00:00<?, ?conversation/s]


Completed: 81902 / 81902 conversations


In [79]:
import json

def save_conversations(conversations, filename="amazon_conversations.jsonl"):

    with open(filename, "w", encoding="utf-8") as f:

        for conversation_id, conversation in enumerate(conversations, start=1):

            record = {
                "conversation_id": conversation_id,
                "messages": conversation
            }

            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    print(f"Saved {len(conversations)} conversations to {filename}")

In [80]:
save_conversations(
    formatted_conversations,
    "/content/amazon_conversations.jsonl"
)

Saved 81902 conversations to /content/amazon_conversations.jsonl


In [81]:
import os

print(
    "File size:",
    os.path.getsize("/content/amazon_conversations.jsonl") / (1024 * 1024),
    "MB"
)

File size: 67.86541652679443 MB


In [82]:
with open("/content/amazon_conversations.jsonl", "r", encoding="utf-8") as f:
    first = json.loads(f.readline())

print(first)

{'conversation_id': 1, 'messages': [{'speaker': 'Customer', 'tweet_id': 1572864, 'text': "@AmazonHelp my order still hasn't arrived3 weeks later,seller is not responding to any emails.I need this cancelled &amp; refunded immediately."}, {'speaker': 'Amazon', 'tweet_id': 1572862, 'text': '@184337 Hi, I am sorry to hear this. You can file a claim with the link provided below http:https://t.co/FjiCD5XSEp ^CR'}, {'speaker': 'Customer', 'tweet_id': 1572863, 'text': "@AmazonHelp This doesn't come up when I click on my order?"}, {'speaker': 'Amazon', 'tweet_id': 1572865, 'text': '@184337 Is this order sold and fulfilled by a third party seller? ^CR'}, {'speaker': 'Customer', 'tweet_id': 1572866, 'text': '@AmazonHelp Yes'}, {'speaker': 'Amazon', 'tweet_id': 1572867, 'text': '@184337 What was the delivery date advised in the order confirmation email ? ^TD'}]}


In [83]:
from google.colab import files

files.download("/content/amazon_conversations.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [91]:
import re
import html
from tqdm.auto import tqdm


def create_rag_documents(formatted_conversations):

    rag_documents = []

    for conversation in tqdm(
        formatted_conversations,
        desc="Creating RAG documents",
        unit="conversation"
    ):

        pairs = []
        current_query = None

        for message in conversation:

            text = str(message["text"])

            # Decode HTML entities
            text = html.unescape(text)

            # Remove @mentions
            text = re.sub(r'@\w+', '', text)

            # Remove URLs
            text = re.sub(r'https?://\S+|http:\S+', '', text)

            # Remove extra whitespace
            text = re.sub(r'\s+', ' ', text)

            text = text.strip()

            if not text:
                continue

            if message["speaker"] == "Customer":

                current_query = text

            elif message["speaker"] == "Amazon" and current_query:

                pairs.append({
                    "customer_query": current_query,
                    "agent_reply": text
                })

                current_query = None

        if pairs:
            rag_documents.append(pairs)

    print(f"\nCreated {len(rag_documents):,} RAG documents.")

    return rag_documents

In [93]:
rag_document = create_rag_documents(formatted_conversations)

Creating RAG documents:   0%|          | 0/81902 [00:00<?, ?conversation/s]


Created 81,473 RAG documents.


In [94]:
rag_document

[[{'customer_query': "my order still hasn't arrived3 weeks later,seller is not responding to any emails.I need this cancelled & refunded immediately.",
   'agent_reply': 'Hi, I am sorry to hear this. You can file a claim with the link provided below ^CR'},
  {'customer_query': "This doesn't come up when I click on my order?",
   'agent_reply': 'Is this order sold and fulfilled by a third party seller? ^CR'},
  {'customer_query': 'Yes',
   'agent_reply': 'What was the delivery date advised in the order confirmation email ? ^TD'}],
 [{'customer_query': 'Amazon Echo届いた',
   'agent_reply': 'ご購入いただきありがとうございました。ぜひご活用くださいませ！😉 EK'}],
 [{'customer_query': 'I think should not promote products with confusing offers like this.. Buy a phone for rupees 2899/- spend 6000/ on service provider- get 1500 discount and claim phone cost is 1399..',
   'agent_reply': "I understand your concern regarding the offer on the product. Pricing and offers are the decision of the sellers. However, I've noted your co

In [96]:
save_conversations(
    rag_document,
    "/content/amazon_conversations_rag_based.jsonl"
)

Saved 81473 conversations to /content/amazon_conversations_rag_based.jsonl


In [97]:
from google.colab import files

files.download("/content/amazon_conversations_rag_based.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>